# Amazon ML Challenge 2026 — Business Entity Resolution

Pipeline: normalize → multi-key IDF blocking → pair features → LightGBM → one-to-one selection.
Settings: **Accelerator: None (CPU)**, **Internet: On**. Add the challenge data as an Input.
Use **Save Version → Save & Run All (Commit)** so it runs in the background (≈1.5–2.5 h).

In [ ]:
# Offline-safe install: use wheels from an attached dataset if present, else PyPI (needs Internet On)
import glob, os, sys, subprocess
whls = glob.glob("/kaggle/input/**/*.whl", recursive=True)
try:
    import unidecode, rapidfuzz
    print("already installed")
except ImportError:
    if whls:
        dirs = sorted({os.path.dirname(w) for w in whls})
        cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-index"] + sum([["--find-links", d] for d in dirs], []) + ["unidecode", "rapidfuzz"]
    else:
        cmd = [sys.executable, "-m", "pip", "install", "-q", "unidecode==1.3.8", "rapidfuzz==3.9.7"]
    print(subprocess.run(cmd, capture_output=True, text=True).stderr[-2000:])
import unidecode, rapidfuzz, lightgbm, pandas, numpy
print("python", sys.version.split()[0], "rapidfuzz", rapidfuzz.__version__, "lightgbm", lightgbm.__version__, "pandas", pandas.__version__, "numpy", numpy.__version__)
!nproc; free -g

In [ ]:
os.makedirs("/kaggle/working/ber/src", exist_ok=True)
%cd /kaggle/working/ber

In [ ]:
%%writefile src/__init__.py
# business entity resolution pipeline

In [ ]:
%%writefile src/blocking.py
"""Candidate generation: IDF-weighted multi-key token blocking, fully vectorized.

Every record emits a set of hashed keys (country-scoped):
  name  : phonetic-skeleton tokens, 5-char prefixes, skeleton token pairs
  addr  : skeleton tokens, (house-number, token) pairs
An S1 record and a pool record share a key -> they accumulate idf(key) into a name or
address score. Keys that are too common (pool df > cap) are dropped. For every S1 we
keep the union of the top-K by total / name / address score. All joins are numpy
range-expansions over sorted key arrays (no Python loop per pair).
"""
import os
import multiprocessing as mp
import numpy as np
import pandas as pd

from .normalize import skeleton

STREET_TYPES = {"st", "ave", "rd", "dr", "ln", "cir", "ct", "blvd", "pl", "pkwy", "hwy", "ter", "trl",
                "sq", "cv", "way", "loop", "n", "s", "e", "w", "ne", "nw", "se", "sw", "ste", "apt", "flr",
                "bldg", "sec", "ngr", "col", "marg", "rue", "all", "chem", "imp", "rte", "fbg", "quai",
                "crs", "res", "main", "new", "old"}
MAX_PAIR_TOKS = 6


MAX_CROSS_NAME, MAX_CROSS_ADDR, MAX_ADDR_PAIR = 4, 6, 6


def record_keys(country, n_clean, a_clean, nums):
    """Returns (name_keys, addr_keys, cross_keys) as sets of strings."""
    c = country
    nk, ak, xk = set(), set(), set()
    ntoks = n_clean.split()
    sk = []
    for t in ntoks:
        s = skeleton(t)
        if len(s) >= 2:
            nk.add(f"t{c}|{s}")
            sk.append(s)
        elif len(t) >= 2:
            nk.add(f"r{c}|{t}")
        if len(t) >= 5:
            nk.add(f"p{c}|{t[:5]}")
    joined = "".join(ntoks)
    if len(joined) >= 8:                       # 'southernexports.com' <-> 'Southern Exports'
        nk.add(f"j{c}|{joined[:8]}")
    sk = list(dict.fromkeys(sk))[:MAX_PAIR_TOKS]
    for i in range(len(sk)):
        for j in range(i + 1, len(sk)):
            a, b = (sk[i], sk[j]) if sk[i] < sk[j] else (sk[j], sk[i])
            nk.add(f"b{c}|{a}|{b}")
    if a_clean or nums:
        ask = []
        for t in a_clean.split():
            if t in STREET_TYPES:
                continue
            s = skeleton(t) if t.isalpha() else t
            if len(s) >= 2:
                ask.append(s)
        ask = list(dict.fromkeys(ask))
        for s in ask:
            ak.add(f"a{c}|{s}")
        nl = nums.split()[:2]
        for n in nl:
            for s in ask:
                ak.add(f"n{c}|{n}|{s}")
        ap = ask[:MAX_ADDR_PAIR]
        for i in range(len(ap)):
            for j in range(i + 1, len(ap)):
                a, b = (ap[i], ap[j]) if ap[i] < ap[j] else (ap[j], ap[i])
                ak.add(f"q{c}|{a}|{b}")
        for s_ in sk[:MAX_CROSS_NAME]:
            for a in ask[:MAX_CROSS_ADDR] + nl:
                xk.add(f"x{c}|{s_}|{a}")
    return nk, ak, xk


def _keys_chunk(args):
    off, countries, names, addrs, nums = args
    recs, keys, fams = [], [], []
    for i, (c, n, a, u) in enumerate(zip(countries, names, addrs, nums)):
        for fam, ks in enumerate(record_keys(c.lower(), n, a, u)):
            for k in ks:
                recs.append(off + i); keys.append(k); fams.append(fam)
    h = pd.util.hash_array(np.array(keys, dtype=object), categorize=False)
    return np.array(recs, np.int32), h, np.array(fams, np.int8)


def build_keys(df, n_jobs=None, chunk=100_000):
    n_jobs = n_jobs or os.cpu_count()
    cols = [df.country.tolist(), df.n_clean.tolist(), df.a_clean.tolist(), df.nums.tolist()]
    parts = [(i, *[c[i:i + chunk] for c in cols]) for i in range(0, len(df), chunk)]
    if n_jobs > 1 and len(parts) > 1:
        with mp.get_context("fork").Pool(n_jobs) as pool:
            res = pool.map(_keys_chunk, parts)
    else:
        res = [_keys_chunk(p) for p in parts]
    return (np.concatenate([r[0] for r in res]), np.concatenate([r[1] for r in res]),
            np.concatenate([r[2] for r in res]))


def _group_rank(g, score):
    """rank (0 = best) of each row within its group g, by descending score."""
    order = np.lexsort((-score, g))
    gs = g[order]
    first = np.r_[0, np.flatnonzero(gs[1:] != gs[:-1]) + 1]
    rank_sorted = np.arange(len(g)) - np.repeat(first, np.diff(np.r_[first, len(g)]))
    rank = np.empty(len(g), np.int32)
    rank[order] = rank_sorted
    return rank


def generate_candidates(s1, pool, cap=600, df_scale=1.0, k_total=30, k_name=10, k_addr=10,
                        budget=60_000_000, n_jobs=None, verbose=True):
    """Returns DataFrame [s1_idx, pool_idx, bs_name, bs_addr, bs_nkeys] (row indices into s1/pool)."""
    import time
    t0 = time.time()
    pr, pk, pf = build_keys(pool, n_jobs)
    sr, sk, sf = build_keys(s1, n_jobs)
    if verbose:
        print(f"[block] keys pool={len(pk):,} s1={len(sk):,} ({time.time() - t0:.0f}s)", flush=True)
    order = np.argsort(pk)
    pk = pk[order]
    pr = pr[order]
    del order, pf
    newk = np.empty(len(pk), bool)
    newk[0] = True
    np.not_equal(pk[1:], pk[:-1], out=newk[1:])
    starts = np.flatnonzero(newk)
    uk = pk[starts]
    del pk, newk
    counts = np.diff(np.r_[starts, len(pr)]).astype(np.int64)
    idf = np.log((len(pool) + 1) / (counts + 1)).astype(np.float32)

    pos = np.searchsorted(uk, sk)
    pos[pos == len(uk)] = 0
    ok = (uk[pos] == sk) & (counts[pos] * df_scale <= cap)
    sr, sf, pos = sr[ok], sf[ok], pos[ok]
    del sk, ok
    if verbose:
        print(f"[block] cap={cap} usable s1 keys={len(sr):,} expansion={counts[pos].sum():,}", flush=True)

    out = []
    # chunk S1 records so each chunk expands to <= budget pairs (bounded memory)
    rec_exp = np.bincount(sr, weights=counts[pos], minlength=len(s1))
    cum = np.cumsum(rec_exp)
    rb = np.unique(np.r_[0, np.searchsorted(cum, np.arange(budget, cum[-1] + budget, budget)), len(s1)])
    bounds = np.searchsorted(sr, rb)
    for b0, b1 in zip(bounds[:-1], bounds[1:]):
        if b1 <= b0:
            continue
        r, f, p = sr[b0:b1], sf[b0:b1], pos[b0:b1]
        cnt = counts[p]
        tot = int(cnt.sum())
        rep_idx = np.repeat(np.arange(len(r)), cnt)
        offs = np.arange(tot) - np.repeat(np.cumsum(cnt) - cnt, cnt) + np.repeat(starts[p], cnt)
        pairs = (r[rep_idx].astype(np.int64) << 24) | pr[offs].astype(np.int64)
        w = idf[p][rep_idx]
        fam = f[rep_idx]
        del offs, rep_idx
        up, inv = np.unique(pairs, return_inverse=True)
        del pairs
        s_name = np.bincount(inv, weights=w * (fam == 0), minlength=len(up)).astype(np.float32)
        s_addr = np.bincount(inv, weights=w * (fam == 1), minlength=len(up)).astype(np.float32)
        s_cross = np.bincount(inv, weights=w * (fam == 2), minlength=len(up)).astype(np.float32)
        nkeys = np.bincount(inv, minlength=len(up)).astype(np.int16)
        g = (up >> 24).astype(np.int32)
        keep = (_group_rank(g, s_name + s_addr + s_cross) < k_total) | (_group_rank(g, s_name) < k_name) | \
               (_group_rank(g, s_addr) < k_addr)
        out.append(pd.DataFrame({"s1_idx": g[keep], "pool_idx": (up[keep] & ((1 << 24) - 1)).astype(np.int32),
                                 "bs_name": s_name[keep], "bs_addr": s_addr[keep], "bs_cross": s_cross[keep],
                                 "bs_nkeys": nkeys[keep]}))
        if verbose:
            print(f"[block] s1 rows {r[0]:,}-{r[-1]:,}: expanded {tot:,} -> uniq {len(up):,} -> kept {keep.sum():,}"
                  f" ({time.time() - t0:.0f}s)", flush=True)
    return pd.concat(out, ignore_index=True)

In [ ]:
%%writefile src/features.py
"""Pairwise features for (S1, pool) candidate pairs.

String features are computed with rapidfuzz in forked worker processes over chunks of
pairs (records are shared copy-on-write via module globals). Context features (ranks /
competition among candidates) are vectorized with pandas groupby.
"""
import math
import os
import multiprocessing as mp
from collections import Counter

import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler, Levenshtein

from .normalize import skeleton

_G = {}  # per-worker state (idf tables)


def build_idf(series_list):
    c = Counter()
    n = 0
    for s in series_list:
        for v in s:
            n += 1
            c.update(set(skeleton(t) for t in v.split()))
    return {k: math.log((n + 1) / (v + 1)) for k, v in c.items()}, math.log(n + 1)


def _skels(s):
    return [skeleton(t) for t in s.split()] if s else []


def _wjacc(a, b, idf, dflt):
    sa, sb = set(a), set(b)
    if not sa or not sb:
        return -1.0, -1.0, 0.0
    inter = sa & sb
    wi = sum(idf.get(t, dflt) for t in inter)
    wu = sum(idf.get(t, dflt) for t in sa | sb)
    # max idf of an S1 token that is missing in the pool record (rare token unexplained)
    miss = max((idf.get(t, dflt) for t in sa - sb), default=0.0)
    return wi / wu if wu else 0.0, len(inter) / len(sa | sb), miss


def _num_feats(n1, n2):
    a, b = n1.split(), n2.split()
    if not a or not b:
        return -1.0, -1.0, -1.0, -1.0
    sa, sb = set(a), set(b)
    inter = len(sa & sb)
    first_eq = float(a[0] == b[0])
    near = 0.0
    if not inter:  # typo'd house number: 4242 vs 424, 6176 vs 1176
        near = float(any(Levenshtein.distance(x, y) <= 1 for x in sa for y in sb if len(x) >= 3 or len(y) >= 3))
    return inter / len(sa | sb), first_eq, float(inter == 0), near


def _pair_feats(n1, a1, u1, st1, l1, n2, a2, u2, st2, l2):
    idf_n, dn, idf_a, da = _G["idf_n"], _G["dn"], _G["idf_a"], _G["da"]
    j1, j2 = n1.replace(" ", ""), n2.replace(" ", "")
    k1, k2 = _skels(n1), _skels(n2)
    wj, jac, miss = _wjacc(k1, k2, idf_n, dn)
    wj_r, _, miss_r = _wjacc(k2, k1, idf_n, dn)
    legal_eq = -1.0 if (not l1 or not l2) else float(l1 == l2)
    legal_ov = -1.0 if (not l1 or not l2) else float(bool(set(l1.split()) & set(l2.split())))
    if a1 and a2:
        ar = fuzz.ratio(a1, a2)
        ats = fuzz.token_set_ratio(a1, a2)
        apr = fuzz.partial_ratio(a1, a2) if len(a1) < 200 and len(a2) < 200 else ar
        awj, ajac, amiss = _wjacc(_skels(a1), _skels(a2), idf_a, da)
        _, _, amiss_r = _wjacc(_skels(a2), _skels(a1), idf_a, da)
    else:
        ar = ats = apr = awj = ajac = amiss = amiss_r = -1.0
    nj, nfe, nconf, nnear = _num_feats(u1, u2)
    st = -1.0 if (not st1 or not st2) else float(st1 == st2)
    return (
        fuzz.ratio(n1, n2), fuzz.token_set_ratio(n1, n2), fuzz.token_sort_ratio(n1, n2),
        fuzz.partial_ratio(n1, n2), JaroWinkler.similarity(j1, j2) if j1 and j2 else 0.0,
        fuzz.ratio(j1, j2), fuzz.partial_ratio(j1, j2) if j1 and j2 else 0.0,
        fuzz.token_set_ratio(" ".join(k1), " ".join(k2)),
        wj, jac, miss, miss_r, float(k1[:1] == k2[:1]),
        len(k1), len(k2), legal_eq, legal_ov,
        ar, ats, apr, awj, ajac, amiss, amiss_r,
        nj, nfe, nconf, nnear, st,
    )


STR_COLS = ["n_ratio", "n_tset", "n_tsort", "n_partial", "n_jw_join", "n_ratio_join", "n_partial_join",
            "n_skel_tset", "n_wjacc", "n_jacc", "n_miss_idf", "n_miss_idf_r", "n_first_eq", "n_ntok1",
            "n_ntok2", "legal_eq", "legal_ov", "a_ratio", "a_tset", "a_partial", "a_wjacc", "a_jacc",
            "a_miss_idf", "a_miss_idf_r", "num_jacc", "num_first_eq", "num_conflict", "num_near", "state_eq"]


STR_SRC = ["n_clean", "a_clean", "nums", "state", "legal"]


def _chunk(cols):
    return np.array([_pair_feats(*r) for r in zip(*cols)], dtype=np.float32)


def _init(idf_n, idf_a):
    _G["idf_n"], _G["dn"] = idf_n
    _G["idf_a"], _G["da"] = idf_a


def string_features(cands, s1, pool, idf_n, idf_a, n_jobs=None, chunk=100_000):
    """Parent slices the needed strings per chunk and ships them to workers, so worker
    memory stays bounded regardless of dataset size."""
    n_jobs = n_jobs or os.cpu_count()
    A = {c: s1[c].values for c in STR_SRC}
    B = {c: pool[c].values for c in STR_SRC}
    ii, jj = cands.s1_idx.values, cands.pool_idx.values

    def parts():
        for k in range(0, len(ii), chunk):
            i, j = ii[k:k + chunk], jj[k:k + chunk]
            yield [A[c][i].tolist() for c in STR_SRC] + [B[c][j].tolist() for c in STR_SRC]

    if n_jobs > 1 and len(ii) > chunk:
        with mp.get_context("fork").Pool(n_jobs, initializer=_init, initargs=(idf_n, idf_a)) as p:
            res = list(p.imap(_chunk, parts(), chunksize=1))
    else:
        _init(idf_n, idf_a)
        res = [_chunk(x) for x in parts()]
    X = np.vstack(res) if res else np.zeros((0, len(STR_COLS)), np.float32)
    return pd.DataFrame(X, columns=STR_COLS, index=cands.index)


def context_features(c, score_col, prefix):
    """Rank / margin of each pair among its S1's candidates and among the pool record's S1s."""
    out = pd.DataFrame(index=c.index)
    g1 = c.groupby("s1_idx")[score_col]
    out[f"{prefix}_rank_s1"] = g1.rank(ascending=False, method="first").astype(np.float32)
    mx = g1.transform("max")
    out[f"{prefix}_gap_s1"] = (mx - c[score_col]).astype(np.float32)
    out[f"{prefix}_ratio_s1"] = (c[score_col] / (mx + 1e-6)).astype(np.float32)
    g2 = c.groupby("pool_idx")[score_col]
    out[f"{prefix}_rank_pool"] = g2.rank(ascending=False, method="first").astype(np.float32)
    mx2 = g2.transform("max")
    out[f"{prefix}_gap_pool"] = (mx2 - c[score_col]).astype(np.float32)
    # margin over the best *other* S1 competing for this pool record
    top1, top2 = _top2(c.pool_idx.values, c[score_col].values)
    v = c[score_col].values
    other = np.where(v >= top1, top2, top1)
    out[f"{prefix}_margin_pool"] = (v - other).astype(np.float32)
    t1, t2 = _top2(c.s1_idx.values, v)
    out[f"{prefix}_top2gap_s1"] = (t1 - t2).astype(np.float32)
    return out


def _top2(g, v):
    """Per-row: best and second-best value within the row's group (second=-1 if none)."""
    order = np.lexsort((-v, g))
    gs, vs = g[order], v[order]
    first = np.r_[True, gs[1:] != gs[:-1]]
    grp = np.cumsum(first) - 1
    starts = np.flatnonzero(first)
    sizes = np.diff(np.r_[starts, len(g)])
    b1 = vs[starts]
    b2 = np.where(sizes > 1, vs[np.minimum(starts + 1, len(g) - 1)], -1.0)
    t1, t2 = np.empty(len(g), vs.dtype), np.empty(len(g), vs.dtype)
    t1[order], t2[order] = b1[grp], b2[grp]
    return t1, t2


def cheap_features(c, s1, pool):
    """Vectorized features that need the FULL candidate set (competition context)."""
    c = c.copy()
    c["bs_total"] = c.bs_name + c.bs_addr + c.bs_cross
    X = pd.DataFrame(index=c.index)
    X["src"] = pool.src.values[c.pool_idx.values].astype(np.float32)
    X["n_cands_s1"] = c.groupby("s1_idx").s1_idx.transform("size").astype(np.float32)
    X["n_cands_pool"] = c.groupby("pool_idx").pool_idx.transform("size").astype(np.float32)
    X["a1_empty"] = (s1.a_clean.values[c.s1_idx.values] == "").astype(np.float32)
    X["a2_empty"] = (pool.a_clean.values[c.pool_idx.values] == "").astype(np.float32)
    for col in ["bs_name", "bs_addr", "bs_cross", "bs_nkeys", "bs_total"]:
        X[col] = c[col].astype(np.float32)
    return pd.concat([X, context_features(c, "bs_total", "bst")], axis=1)


def all_features(cands, s1, pool, idf_n, idf_a, n_jobs=None):
    X = cheap_features(cands, s1, pool)
    S = string_features(cands, s1, pool, idf_n, idf_a, n_jobs)
    return pd.concat([S, X], axis=1)

In [ ]:
%%writefile src/matching.py
"""Pair classifier, set selection, and the competition metric."""
import numpy as np
import pandas as pd
import lightgbm as lgb

NON_FEATS = {"y", "s1_idx", "pool_idx", "fold"}

LGB_PARAMS = dict(objective="binary", learning_rate=0.08, num_leaves=127, min_data_in_leaf=100,
                  feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                  max_bin=255, verbose=-1, num_threads=0)


def feat_cols(X):
    return [c for c in X.columns if c not in NON_FEATS]


def train_lgb(X, y, rounds=600, params=None, valid=None):
    p = dict(LGB_PARAMS, **(params or {}))
    dtr = lgb.Dataset(X, y, free_raw_data=True)
    vs = [lgb.Dataset(valid[0], valid[1], reference=dtr)] if valid is not None else []
    cb = [lgb.log_evaluation(100)] + ([lgb.early_stopping(50, verbose=False)] if vs else [])
    return lgb.train(p, dtr, num_boost_round=rounds, valid_sets=vs, callbacks=cb)


def select(s1_idx, pool_idx, p, thr=0.5, one_to_one=True, min_top=None):
    """Pick matches. Each pool record goes to at most one S1 (its best-scoring one).
    `min_top`: if set, an S1 whose best candidate has p >= min_top keeps it even if < thr."""
    df = pd.DataFrame({"s1": s1_idx, "pool": pool_idx, "p": p})
    if one_to_one:
        best = df.groupby("pool").p.transform("max")
        df = df[df.p >= best]
        df = df.drop_duplicates("pool")
    keep = df.p >= thr
    if min_top is not None:
        top = df.groupby("s1").p.transform("max")
        keep |= (df.p >= top) & (df.p >= min_top)
    return df[keep][["s1", "pool", "p"]]


def f05_macro(pred_pairs, true_pairs, all_s1):
    """pred_pairs/true_pairs: DataFrames with columns s1, pool (ids or idx). all_s1: iterable of s1 keys."""
    all_s1 = pd.Index(pd.unique(np.asarray(all_s1)))
    tp = pred_pairs.merge(true_pairs, on=["s1", "pool"]).groupby("s1").size()
    npred = pred_pairs.groupby("s1").size()
    ntrue = true_pairs.groupby("s1").size()
    tp, npred, ntrue = (s.reindex(all_s1, fill_value=0).values.astype(float) for s in (tp, npred, ntrue))
    with np.errstate(divide="ignore", invalid="ignore"):
        prec = np.where(npred > 0, tp / npred, 0.0)
        rec = np.where(ntrue > 0, tp / ntrue, 0.0)
        f = np.where(tp > 0, 1.25 * prec * rec / (0.25 * prec + rec), 0.0)
    f = np.where((ntrue == 0) & (npred == 0), 1.0, f)
    return float(f.mean()), dict(precision=float(prec[npred > 0].mean()) if (npred > 0).any() else 0.0,
                                 recall=float(rec[ntrue > 0].mean()),
                                 singleton_acc=float(((npred == 0)[ntrue == 0]).mean()) if (ntrue == 0).any() else 1.0)

In [ ]:
%%writefile src/normalize.py
"""Text normalization for business names and addresses.

Country-agnostic: every rule is language-level (Latin transliteration, punctuation,
abbreviation folding), never keyed on a fixed country list, so unseen countries
(France in test) go through the same path.
"""
import re
from unidecode import unidecode

# ----------------------------------------------------------------------------- tokens
LEGAL = {
    # EN / IN
    "pvt": "pvt", "private": "pvt", "pte": "pvt", "prvt": "pvt",
    "ltd": "ltd", "limited": "ltd", "ltda": "ltd", "lt": "ltd",
    "llc": "llc", "llp": "llp", "lp": "lp", "plc": "plc", "pllc": "pllc", "pc": "pc",
    "inc": "inc", "incorporated": "inc", "corp": "corp", "corporation": "corp",
    "co": "co", "company": "co", "cos": "co", "tbk": "tbk", "gmbh": "gmbh", "ag": "ag",
    "public": "public", "opc": "opc", "holdings": None, "group": None,
    # FR (and generic civil-law)
    "sarl": "sarl", "sas": "sas", "sasu": "sasu", "eurl": "eurl", "sa": "sa", "sci": "sci",
    "snc": "snc", "sca": "sca", "scop": "scop", "selarl": "selarl", "scp": "scp", "gie": "gie",
}
LEGAL_FORMS = {v for v in LEGAL.values() if v}
NAME_STOP = {"the", "and", "of", "de", "des", "du", "la", "le", "les", "d", "l", "et", "a",
             "an", "www", "com", "net", "org", "in", "co.in", "biz", "info", "fr", "us", "dba",
             "nee", "id", "m/s", "ms", "mr", "mrs", "sri", "shri", "shree"}

ADDR_MAP = {
    "street": "st", "str": "st", "st": "st", "avenue": "ave", "av": "ave", "ave": "ave", "avn": "ave",
    "road": "rd", "rd": "rd", "drive": "dr", "dr": "dr", "lane": "ln", "ln": "ln",
    "circle": "cir", "cir": "cir", "court": "ct", "ct": "ct", "boulevard": "blvd", "blvd": "blvd",
    "bd": "blvd", "bld": "blvd", "place": "pl", "pl": "pl", "parkway": "pkwy", "pkwy": "pkwy",
    "highway": "hwy", "hwy": "hwy", "terrace": "ter", "ter": "ter", "trail": "trl", "trl": "trl",
    "square": "sq", "sq": "sq", "cove": "cv", "cv": "cv", "way": "way", "loop": "loop",
    "north": "n", "south": "s", "east": "e", "west": "w", "northeast": "ne", "northwest": "nw",
    "southeast": "se", "southwest": "sw", "suite": "ste", "ste": "ste", "apartment": "apt", "apt": "apt",
    "floor": "flr", "flr": "flr", "fl": "flr", "etage": "flr", "building": "bldg", "bldg": "bldg",
    "sector": "sec", "sec": "sec", "nagar": "ngr", "ngr": "ngr", "colony": "col", "marg": "marg",
    # FR
    "rue": "rue", "r": "rue", "allee": "all", "all": "all", "chemin": "chem", "chem": "chem",
    "impasse": "imp", "imp": "imp", "route": "rte", "rte": "rte", "faubourg": "fbg", "fbg": "fbg",
    "quai": "quai", "cours": "crs", "crs": "crs", "residence": "res", "res": "res",
}
ADDR_STOP = {"no", "num", "number", "nr", "of", "the", "de", "des", "du", "la", "le", "les", "d", "l",
             "city", "cdp", "borough", "village", "town", "township", "bis", "ter", "unit", "near",
             "opp", "opposite", "behind", "landmark", "house", "door", "flat", "plot", "shop", "h",
             "hno", "dno", "p", "s/o", "so", "at", "po", "post", "dist", "district", "tehsil", "taluka",
             "eme", "er", "nd", "rd_", "th"}

# state / region names -> canonical code, matched by phonetic skeleton so that
# "Maharashtra", "महाराष्ट्र" (-> "mhaaraassttr") and typos land on the same code.
_STATES = {
    # US
    "al": "alabama", "ak": "alaska", "az": "arizona", "ar": "arkansas", "ca": "california",
    "co": "colorado", "ct": "connecticut", "de": "delaware", "dc": "district of columbia",
    "fl": "florida", "ga": "georgia", "hi": "hawaii", "id": "idaho", "il": "illinois",
    "in": "indiana", "ia": "iowa", "ks": "kansas", "ky": "kentucky", "la": "louisiana",
    "me": "maine", "md": "maryland", "ma": "massachusetts", "mi": "michigan", "mn": "minnesota",
    "ms": "mississippi", "mo": "missouri", "mt": "montana", "ne": "nebraska", "nv": "nevada",
    "nh": "new hampshire", "nj": "new jersey", "nm": "new mexico", "ny": "new york",
    "nc": "north carolina", "nd": "north dakota", "oh": "ohio", "ok": "oklahoma", "or": "oregon",
    "pa": "pennsylvania", "ri": "rhode island", "sc": "south carolina", "sd": "south dakota",
    "tn": "tennessee", "tx": "texas", "ut": "utah", "vt": "vermont", "va": "virginia",
    "wa": "washington", "wv": "west virginia", "wi": "wisconsin", "wy": "wyoming",
}
_IN_STATES = {
    "mh": "maharashtra", "dl": "delhi", "up": "uttar pradesh", "ka": "karnataka", "tn": "tamil nadu",
    "gj": "gujarat", "wb": "west bengal", "tg": "telangana", "ts": "telangana", "hr": "haryana",
    "kl": "kerala", "rj": "rajasthan", "br": "bihar", "mp": "madhya pradesh", "ap": "andhra pradesh",
    "od": "orissa", "or": "odisha", "pb": "punjab", "jh": "jharkhand", "ch": "chandigarh",
    "ga": "goa", "as": "assam", "uk": "uttarakhand", "hp": "himachal pradesh", "jk": "jammu and kashmir",
    "ct": "chhattisgarh", "cg": "chhattisgarh", "py": "puducherry",
}

# ----------------------------------------------------------------------------- helpers
_RE_ID = re.compile(r"\(\s*id\s*:?\s*\d+\s*\)", re.I)
_RE_DOT_SPLIT = re.compile(r"\.(?=[a-z0-9]{2,})")
_RE_NONALNUM = re.compile(r"[^a-z0-9/ ]+")
_RE_SPACES = re.compile(r"\s+")
_RE_ORD = re.compile(r"\b(\d+)(st|nd|rd|th|eme|er|e)\b")
_RE_NUM = re.compile(r"\d+")
_OCR = str.maketrans({"0": "o", "1": "l", "5": "s", "3": "e", "4": "a", "8": "b", "7": "t"})


def to_ascii(s: str) -> str:
    if not s:
        return ""
    if not s.isascii():
        s = unidecode(s.replace("\u00b0", " "))
    return s.lower()


def _basic(s: str) -> str:
    s = to_ascii(s)
    s = _RE_ID.sub(" ", s)
    s = s.replace("&", " and ").replace("+", " plus ").replace("@", " at ")
    s = _RE_DOT_SPLIT.sub(" ", s)      # "pvt.ltd" -> "pvt ltd", "ironl.com" -> "ironl com"
    s = s.replace(".", "")             # "s.a.r.l." -> "sarl", "inc." -> "inc"
    s = s.replace("'", "")
    s = _RE_NONALNUM.sub(" ", s)
    return _RE_SPACES.sub(" ", s).strip()


def collapse(t: str) -> str:
    out = []
    for c in t:
        if not out or out[-1] != c:
            out.append(c)
    return "".join(out)


_DIGRAPHS = [("ph", "f"), ("bh", "b"), ("kh", "k"), ("gh", "g"), ("th", "t"), ("dh", "d"),
             ("sh", "s"), ("ch", "c"), ("ck", "k"), ("nb", "mb"), ("np", "mp"), ("x", "ks"),
             ("q", "k"), ("c", "k"), ("v", "b"), ("w", ""), ("z", "j"), ("g", "j"), ("h", "")]
_VOWELS = set("aeiouy")


def skeleton(t: str) -> str:
    """Phonetic consonant skeleton; makes transliterations / vowel typos collide.
    'private'/'praaivett'/'praibhet' -> 'prbt';  'management'/'myaanejmentt' -> 'mnjmnt'."""
    t = collapse(t)
    for a, b in _DIGRAPHS:
        if a in t:
            t = t.replace(a, b)
    return collapse("".join(c for c in t if c not in _VOWELS))


def _fix_ocr(tok: str) -> str:
    # tokens mixing letters and digits with a majority of letters: '0ne' -> 'one'
    if tok.isalnum() and not tok.isalpha() and not tok.isdigit():
        n_alpha = sum(c.isalpha() for c in tok)
        if n_alpha >= len(tok) / 2:
            return tok.translate(_OCR)
    return tok


_STATE_SKEL = {}
for _code, _name in list(_STATES.items()) + list(_IN_STATES.items()):
    _STATE_SKEL.setdefault(skeleton(_name.replace(" ", "")), _code)


def state_of(part: str):
    """Map one comma-separated address component to a state code if it is one."""
    p = part.replace(" ", "")
    if len(p) == 2 and p.isalpha():
        return p if (p in _STATES or p in _IN_STATES) else None
    return _STATE_SKEL.get(skeleton(p)) if len(p) >= 4 else None


# ----------------------------------------------------------------------------- public API
_LEGAL_SKEL = {skeleton(k): v for k, v in LEGAL.items() if v and len(k) >= 5}


def _legal(t: str):
    """Legal-form lookup that survives transliteration ('praivet', 'limittedd')."""
    if t in LEGAL:
        return LEGAL[t]
    if len(t) >= 5:
        c = collapse(t)
        if c in LEGAL:
            return LEGAL[c]
        return _LEGAL_SKEL.get(skeleton(t))
    return None


def norm_name(s: str):
    """Returns (clean_string, core_tokens, legal_forms)."""
    b = _basic(s)
    toks = [_fix_ocr(t) for t in b.replace("/", " ").split()]
    core, legal = [], []
    for t in toks:
        v = _legal(t)
        if v:
            legal.append(v)
            continue
        if t in NAME_STOP:
            continue
        core.append(collapse(t) if len(t) > 3 else t)
    return " ".join(core), core, sorted(set(legal))


def norm_addr(s: str):
    """Returns (clean_string, alpha_tokens, numbers, state_code)."""
    if not s:
        return "", [], [], ""
    a = to_ascii(s).replace("#", " ")
    state, keep = "", []
    for part in a.split(","):
        p = _RE_NONALNUM.sub(" ", part.replace(".", " ")).strip()
        st = state_of(p) if p else None
        if st:
            state = st
        else:
            keep.append(part)
    b = _basic(",".join(keep))
    b = _RE_ORD.sub(r"\1", b)
    b = b.replace("/", " ")
    alpha, nums = [], []
    for t in b.split():
        if t.isdigit():
            t = t.lstrip("0") or "0"
            nums.append(t)
            continue
        t = _fix_ocr(t)
        if any(c.isdigit() for c in t):          # 1715b, 8c5 -> keep leading number too
            m = _RE_NUM.match(t)
            if m:
                nums.append(m.group(0).lstrip("0") or "0")
            alpha.append(t)
            continue
        t = ADDR_MAP.get(t, t)
        if t in ADDR_STOP or len(t) < 2:
            continue
        alpha.append(collapse(t))
    return " ".join(alpha), alpha, nums, state

In [ ]:
%%writefile src/prep.py
"""Load the three sources, normalize in parallel, cache as parquet."""
import os
import multiprocessing as mp
import numpy as np
import pandas as pd

from .normalize import norm_name, norm_addr

READ_KW = dict(sep="\t", dtype=str, keep_default_na=False, quoting=3)


def read_tsv(path):
    df = pd.read_csv(path, **READ_KW)
    df.columns = [c.strip() for c in df.columns]
    for c in df.columns:
        df[c] = df[c].str.replace("\r", "", regex=False)
    return df


def _norm_chunk(args):
    names, addrs = args
    out = []
    for n, a in zip(names, addrs):
        nc, _, legal = norm_name(n)
        ac, _, nums, st = norm_addr(a)
        out.append((nc, " ".join(legal), ac, " ".join(nums), st))
    return out


def normalize_df(df, n_jobs=None, chunk=50_000):
    n_jobs = n_jobs or os.cpu_count()
    names, addrs = df.business_name.tolist(), df.business_address.tolist()
    parts = [(names[i:i + chunk], addrs[i:i + chunk]) for i in range(0, len(df), chunk)]
    if n_jobs > 1 and len(parts) > 1:
        with mp.get_context("fork").Pool(n_jobs) as pool:
            res = pool.map(_norm_chunk, parts)
    else:
        res = [_norm_chunk(p) for p in parts]
    flat = [r for part in res for r in part]
    out = pd.DataFrame(flat, columns=["n_clean", "legal", "a_clean", "nums", "state"], index=df.index)
    # raw text is no longer needed downstream; dropping it roughly halves RAM at full scale
    return pd.concat([df.drop(columns=["business_name", "business_address"]), out], axis=1)


def load_split(data_dir, split, cache_dir=None, n_jobs=None):
    """Returns (s1, pool). pool = S2 ++ S3 with a `src` column (2/3)."""
    cache = os.path.join(cache_dir, f"{split}_norm") if cache_dir else None
    if cache and os.path.exists(cache + "_s1.parquet"):
        return pd.read_parquet(cache + "_s1.parquet"), pd.read_parquet(cache + "_pool.parquet")
    d = os.path.join(data_dir, split)
    s1 = read_tsv(os.path.join(d, f"{split}_source1.tsv"))
    s2 = read_tsv(os.path.join(d, f"{split}_source2.tsv"))
    s3 = read_tsv(os.path.join(d, f"{split}_source3.tsv"))
    s2["src"], s3["src"] = np.int8(2), np.int8(3)
    pool = pd.concat([s2, s3], ignore_index=True)
    s1["src"] = np.int8(1)
    s1 = normalize_df(s1, n_jobs)
    pool = normalize_df(pool, n_jobs)
    if cache:
        os.makedirs(cache_dir, exist_ok=True)
        s1.to_parquet(cache + "_s1.parquet")
        pool.to_parquet(cache + "_pool.parquet")
    return s1, pool


def load_gt(path):
    gt = read_tsv(path)
    rows = [(s, m) for s, ms in zip(gt.source1_entity_id, gt.matched_entity_ids) for m in ms.split(",") if m]
    return gt, pd.DataFrame(rows, columns=["s1_id", "pool_id"])

In [ ]:
%%writefile src/run.py
"""End-to-end pipeline: data -> normalize -> blocking -> features -> LightGBM -> selection -> TSVs.

python -m src.run --data <.../student_resource/dataset> --work <cache dir> --out <output dir>

Every stage caches to --work, so a re-run skips finished stages (delete a file to redo it).
"""
import argparse
import gc
import json
import os
import pickle
import time

import numpy as np
import pandas as pd

from .prep import load_split, load_gt
from .blocking import generate_candidates
from .features import build_idf, cheap_features, string_features
from .matching import train_lgb, feat_cols, select, f05_macro

T0 = time.time()


def log(*a):
    print(f"[{time.time() - T0:7.0f}s]", *a, flush=True)


def cached(path, fn):
    if os.path.exists(path):
        log("cache hit", os.path.basename(path))
        return pd.read_parquet(path)
    df = fn()
    df.to_parquet(path)
    return df


def s1_hash(ids):
    return (pd.util.hash_array(np.asarray(ids, dtype=object)) % 10_000).astype(np.int32)


def labels(c, s1, pool, gtp):
    k = pd.MultiIndex.from_arrays([s1.entity_id.values[c.s1_idx.values], pool.entity_id.values[c.pool_idx.values]])
    return k.isin(pd.MultiIndex.from_arrays([gtp.s1_id.values, gtp.pool_id.values])).astype(np.int8)


def write_lists(path, s1_ids, pairs_s1, pairs_pool, col):
    """One row per S1 (empty list allowed), ids comma-joined, no duplicates."""
    d = pd.DataFrame({"s1": pairs_s1, "pool": pairs_pool}).drop_duplicates()
    lists = d.groupby("s1").pool.agg(",".join)
    out = pd.DataFrame({"source1_entity_id": s1_ids})
    out[col] = out.source1_entity_id.map(lists).fillna("")
    out.to_csv(path, sep="\t", index=False)
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--work", default="work")
    ap.add_argument("--out", default="output")
    ap.add_argument("--train-frac", type=float, default=0.30, help="fraction of train S1 used to fit the model")
    ap.add_argument("--holdout", type=float, default=0.2, help="share of that sample held out for threshold tuning")
    ap.add_argument("--cap", type=int, default=600)
    ap.add_argument("--df-scale", type=float, default=1.0)
    ap.add_argument("--k-total", type=int, default=30)
    ap.add_argument("--k-name", type=int, default=10)
    ap.add_argument("--k-addr", type=int, default=10)
    ap.add_argument("--rounds", type=int, default=1500)
    ap.add_argument("--threshold", type=float, default=None, help="override the tuned threshold")
    ap.add_argument("--predict-chunk", type=int, default=4_000_000)
    ap.add_argument("--jobs", type=int, default=None)
    a = ap.parse_args()
    os.makedirs(a.work, exist_ok=True)
    os.makedirs(a.out, exist_ok=True)
    W = lambda f: os.path.join(a.work, f)
    bkw = dict(cap=a.cap, df_scale=a.df_scale, k_total=a.k_total, k_name=a.k_name, k_addr=a.k_addr, n_jobs=a.jobs)

    # ------------------------------------------------------------------ TRAIN
    model_path = W("model.pkl")
    if not os.path.exists(model_path):
        s1, pool = load_split(a.data, "train", a.work, a.jobs)
        log("train loaded", len(s1), len(pool))
        _, gtp = load_gt(os.path.join(a.data, "train", "train_ground_truth.tsv"))
        c = cached(W("train_cands.parquet"), lambda: generate_candidates(s1, pool, **bkw))
        y_all = labels(c, s1, pool, gtp)
        log(f"train cands {len(c):,}  per S1 {len(c) / len(s1):.1f}  "
            f"blocking recall {y_all.sum() / len(gtp):.4f}")
        # competition context needs the FULL candidate set -> compute before sampling
        cheap = cheap_features(c, s1, pool)
        h = s1_hash(s1.entity_id.values)[c.s1_idx.values]
        use = h < a.train_frac * 10_000
        c, cheap, y = c[use].reset_index(drop=True), cheap[use].reset_index(drop=True), y_all[use]
        h = h[use]
        del y_all
        gc.collect()
        idf_n = build_idf([s1.n_clean, pool.n_clean])
        idf_a = build_idf([s1.a_clean, pool.a_clean])
        log(f"string features on {len(c):,} sampled pairs")
        X = pd.concat([string_features(c, s1, pool, idf_n, idf_a, a.jobs), cheap], axis=1)
        del cheap
        F = feat_cols(X)
        va = h >= (a.train_frac * (1 - a.holdout)) * 10_000
        log(f"fit lgb: train {(~va).sum():,} rows, holdout {va.sum():,} rows, pos rate {y.mean():.3f}")
        m = train_lgb(X.loc[~va, F], y[~va], rounds=a.rounds, valid=(X.loc[va, F], y[va]))
        p = m.predict(X.loc[va, F], num_threads=0)
        # threshold tuning on holdout S1s against the FULL ground truth (blocking misses count)
        hs1 = np.unique(c.s1_idx.values[va])
        ids = s1.entity_id.values
        true = gtp[gtp.s1_id.isin(set(ids[hs1]))].rename(columns={"s1_id": "s1", "pool_id": "pool"})
        best = (-1, 0.5)
        for thr in np.arange(0.30, 0.92, 0.04):
            sel = select(c.s1_idx.values[va], c.pool_idx.values[va], p, thr)
            pred = pd.DataFrame({"s1": ids[sel.s1.values], "pool": pool.entity_id.values[sel.pool.values]})
            f, info = f05_macro(pred, true, ids[hs1])
            log(f"  thr {thr:.2f}  F0.5 {f:.4f}  {info}")
            best = max(best, (f, thr))
        log(f"HOLDOUT F0.5 {best[0]:.4f} at threshold {best[1]:.2f}")
        imp = pd.Series(m.feature_importance("gain"), F).sort_values(ascending=False)
        log("top features", imp.head(15).round(0).to_dict())
        with open(model_path, "wb") as f:
            pickle.dump(dict(model=m, feats=F, thr=float(best[1]), holdout_f05=best[0]), f)
        json.dump(dict(holdout_f05=best[0], threshold=float(best[1]), best_iter=m.best_iteration,
                       features=F), open(W("train_report.json"), "w"), indent=1)
        del X, c, s1, pool
        gc.collect()

    with open(model_path, "rb") as f:
        M = pickle.load(f)
    thr = a.threshold if a.threshold is not None else M["thr"]
    log(f"using threshold {thr:.2f}")

    # ------------------------------------------------------------------ TEST
    s1, pool = load_split(a.data, "test", a.work, a.jobs)
    log("test loaded", len(s1), len(pool), s1.country.value_counts().to_dict())
    c = cached(W("test_cands.parquet"), lambda: generate_candidates(s1, pool, **bkw))
    log(f"test cands {len(c):,}  per S1 {len(c) / len(s1):.1f}")
    write_lists(os.path.join(a.out, "candidate_pairs.tsv"), s1.entity_id.values,
                s1.entity_id.values[c.s1_idx.values], pool.entity_id.values[c.pool_idx.values],
                "candidate_entity_ids")
    cheap = cheap_features(c, s1, pool)
    idf_n = build_idf([s1.n_clean, pool.n_clean])
    idf_a = build_idf([s1.a_clean, pool.a_clean])
    p = np.zeros(len(c), np.float32)
    for k in range(0, len(c), a.predict_chunk):
        sl = slice(k, k + a.predict_chunk)
        cc = c.iloc[sl]
        X = pd.concat([string_features(cc, s1, pool, idf_n, idf_a, a.jobs), cheap.iloc[sl]], axis=1)
        p[sl] = M["model"].predict(X[M["feats"]], num_threads=0)
        log(f"  predicted {min(k + a.predict_chunk, len(c)):,}/{len(c):,}")
    pd.DataFrame({"s1_idx": c.s1_idx.values, "pool_idx": c.pool_idx.values, "p": p}).to_parquet(W("test_scores.parquet"))
    sel = select(c.s1_idx.values, c.pool_idx.values, p, thr)
    out = write_lists(os.path.join(a.out, "matching_results.tsv"), s1.entity_id.values,
                      s1.entity_id.values[sel.s1.values], pool.entity_id.values[sel.pool.values],
                      "matched_entity_ids")
    n = out.matched_entity_ids.str.len().gt(0)
    log(f"DONE: {len(out):,} S1 rows, {n.mean():.3f} with >=1 match, {len(sel):,} matched pairs")
    log("by country (share with match):", out.assign(c=s1.country.values, n=n).groupby("c").n.mean().round(3).to_dict())


if __name__ == "__main__":
    main()

In [ ]:
hits = glob.glob("/kaggle/input/**/train_source1.tsv", recursive=True)
assert hits, "Add the challenge dataset as an Input first"
DATA = os.path.dirname(os.path.dirname(hits[0]))   # .../dataset
ROOT = os.path.dirname(DATA)                       # .../student_resource
print(DATA)

In [ ]:
!python -m src.run --data "$DATA" --work /kaggle/working/work --out /kaggle/working/output 2>&1 | grep -v "Warning\]"

In [ ]:
!python "$ROOT/utils/validate_submission.py" --matching /kaggle/working/output/matching_results.tsv \
    --candidate /kaggle/working/output/candidate_pairs.tsv --test-dir "$DATA/test"
!cat /kaggle/working/work/train_report.json | head -5

In [ ]:
# free disk: keep outputs + model/report, drop big caches
!rm -f /kaggle/working/work/*_norm_*.parquet /kaggle/working/work/*_cands.parquet
!ls -la /kaggle/working/output /kaggle/working/work